In [ ]:
# Cell 0 — mandatory session context and input audit
from pathlib import Path
import csv, json, os, re

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
print("Repository root:", ROOT)
print("notebooks/ listing:")
for p in sorted(Path("notebooks").iterdir()):
    print(" -", p.name)
preferred = Path("notebooks/22_anchor_facilities.ipynb")
if preferred.exists():
    print("NOTEBOOK NUMBER CHECK: 22_anchor_facilities.ipynb exists; this running notebook is using 22.")
else:
    taken = {int(m.group(1)) for p in Path("notebooks").glob("*.ipynb") if (m:=re.match(r"(\d+)", p.name))}
    n = 22
    while n in taken:
        n += 1
    print(f"NOTEBOOK NUMBER CHECK: 22_anchor_facilities.ipynb is free. Use notebook number {n}.")

with open("terra-app/TERRA_build_log.md", encoding="utf-8") as f:
    build_log = f.read()
s0 = build_log[build_log.index("Phase S0"):build_log.index("Codex verification protocol", build_log.index("Phase S0"))]
print("\nS0 orchestration entry loaded; binding length:", len(s0))

with open("data/terra_action_taxonomy.md", encoding="utf-8") as f:
    taxonomy_md = f.read()
sec62 = taxonomy_md[taxonomy_md.index("### 6.2 Economic Capital"):taxonomy_md.index("### 6.3", taxonomy_md.index("### 6.2"))]
print("terra_action_taxonomy.md §6.2 loaded:")
print(sec62.strip())

study_rows = list(csv.DictReader(open("data/processed/mw_study_counties.csv", encoding="utf-8")))
print("\nmw_study_counties.csv rows:", len(study_rows), "CRS: non-spatial CSV")
for gp in ["data/processed/mw_counties.geojson", "data/processed/power_plants_with_ba.geojson"]:
    gj = json.load(open(gp, encoding="utf-8"))
    crs = gj.get("crs", {}).get("properties", {}).get("name", "unspecified; assumed CRS84 lon/lat")
    print(f"{gp} rows: {len(gj['features'])}; CRS: {crs}")

cards = json.load(open("data/processed/mw_county_cards.json", encoding="utf-8"))
flagship_total = sum(len(c.get("flagship_assets", [])) for c in cards.values())
print("mw_county_cards.json county records:", len(cards), "flagship assets:", flagship_total)
print("Flagship assets block:")
for geoid, card in cards.items():
    for asset in card.get("flagship_assets", []):
        print(geoid, asset.get("name"), asset.get("type"), asset.get("capacity_or_load_mw"), asset.get("source_url"))

In [ ]:
# Cell 1 — imports, constants, geometry helpers
from pathlib import Path
import csv, io, json, math, os, re, time, urllib.request, urllib.error, zipfile, subprocess
from collections import defaultdict, Counter
from datetime import date
from difflib import SequenceMatcher

RAW = Path("data/raw")
PROCESSED = Path("data/processed")
RAW.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)
TODAY = date.today().isoformat()
GHGRP_YEAR = 2023
QCEW_YEAR = 2024

STATE_FIPS = {"CO":"08", "MT":"30", "WY":"56"}
STATE_NAME = {"CO":"Colorado", "MT":"Montana", "WY":"Wyoming"}
WY_COUNTIES = {r["GEOID"] for r in study_rows if r["state"] == "WY"}
STUDY_GEOIDS = {r["GEOID"] for r in study_rows}
STUDY_BY_GEOID = {r["GEOID"]: r for r in study_rows}

def slurp_url(url, timeout=90):
    req = urllib.request.Request(url, headers={"User-Agent":"TERRA-F0-anchor-notebook/1.0"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.read(), dict(r.headers), r.status

def cache_url(url, path, binary=False, timeout=90):
    path = Path(path)
    if path.exists() and path.stat().st_size > 0:
        data = path.read_bytes()
        return data, "cache"
    data, headers, status = slurp_url(url, timeout=timeout)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_bytes(data)
    return data, f"http_{status}"

def read_csv_bytes(data, delimiter=','):
    txt = data.decode("utf-8-sig", "replace")
    return list(csv.DictReader(io.StringIO(txt), delimiter=delimiter))

def clean(v):
    if v is None: return None
    return str(v).strip().strip('"').strip()

def to_float(v):
    try:
        s = clean(v)
        return None if s in (None, "", "None", "nan") else float(s)
    except Exception:
        return None

def to_int(v):
    x = to_float(v)
    return None if x is None else int(round(x))

def ring_contains(ring, x, y):
    inside = False
    n = len(ring)
    for i in range(n):
        x1, y1 = ring[i][0], ring[i][1]
        x2, y2 = ring[(i+1)%n][0], ring[(i+1)%n][1]
        if ((y1 > y) != (y2 > y)):
            xin = (x2-x1) * (y-y1) / ((y2-y1) or 1e-30) + x1
            if x < xin:
                inside = not inside
    return inside

def geom_contains(geom, x, y):
    if x is None or y is None or not geom:
        return False
    polys = [geom["coordinates"]] if geom["type"] == "Polygon" else geom["coordinates"]
    for poly in polys:
        if not poly: continue
        if ring_contains(poly[0], x, y) and not any(ring_contains(hole, x, y) for hole in poly[1:]):
            return True
    return False

def geom_centroid(geom):
    xs=[]; ys=[]
    polys = [geom["coordinates"]] if geom["type"] == "Polygon" else geom["coordinates"]
    for poly in polys:
        for x,y,*_ in poly[0]:
            xs.append(x); ys.append(y)
    return (sum(xs)/len(xs), sum(ys)/len(ys)) if xs else (None, None)

def county_lookup_for_point(x, y, counties):
    for geoid, rec in counties.items():
        if geom_contains(rec["geometry"], x, y):
            return geoid
    return None

counties_gj = json.load(open("data/processed/mw_counties.geojson", encoding="utf-8"))
COUNTY_GEOMS = {f["properties"]["GEOID"]:{"properties":f["properties"],"geometry":f["geometry"],"centroid":geom_centroid(f["geometry"])} for f in counties_gj["features"]}
print("Helpers ready. Study counties:", len(STUDY_GEOIDS), "WY counties:", len(WY_COUNTIES))

In [ ]:
# Cell 2 — source pulls: EPA GHGRP, MSHA, BLS QCEW; keyed sources logged when unavailable
manual_notes = []

# EPA Envirofacts endpoint verification and GHGRP pulls.
endpoint_probe = "https://data.epa.gov/efservice/PUB_DIM_FACILITY/STATE/WY/ROWS/0:1/CSV"
try:
    probe_data, probe_headers, probe_status = slurp_url(endpoint_probe, timeout=60)
    ghgrp_source = "EPA Envirofacts GHGRP PUB_DIM_FACILITY + PUB_FACTS_SECTOR_GHG_EMISSION"
    print("GHGRP source used:", ghgrp_source)
    print("Endpoint verified:", endpoint_probe, "status", probe_status)
except Exception as exc:
    ghgrp_source = "unavailable"
    manual_notes.append(f"{TODAY}: GHGRP Envirofacts endpoint probe failed: {exc}. Manual source: https://data.epa.gov/efservice/ or EPA FLIGHT bulk export.")
    print("GHGRP source used:", ghgrp_source, exc)

ghgrp_facility_rows = []
if ghgrp_source != "unavailable":
    for st in ["WY", "CO", "MT"]:
        url = f"https://data.epa.gov/efservice/PUB_DIM_FACILITY/STATE/{st}/CSV"
        raw_path = RAW / f"ghgrp_pub_dim_facility_{st}_{GHGRP_YEAR}_pulled_{TODAY}.csv"
        data, how = cache_url(url, raw_path, timeout=120)
        rows = read_csv_bytes(data)
        ghgrp_facility_rows.extend(rows)
        print(f"GHGRP facility dimension {st}: {len(rows)} rows ({how}) -> {raw_path}")
    # Latest populated emissions year. 2024 is commonly unavailable early; 2023 is the latest verified here.
    url = f"https://data.epa.gov/efservice/PUB_FACTS_SECTOR_GHG_EMISSION/YEAR/{GHGRP_YEAR}/CSV"
    raw_path = RAW / f"ghgrp_pub_facts_sector_ghg_emission_{GHGRP_YEAR}_pulled_{TODAY}.csv"
    data, how = cache_url(url, raw_path, timeout=180)
    ghgrp_emission_rows = read_csv_bytes(data)
    print(f"GHGRP emissions {GHGRP_YEAR}: {len(ghgrp_emission_rows)} rows ({how}) -> {raw_path}")
else:
    ghgrp_emission_rows = []

# MSHA active mines + yearly production/employment.
msha_sources = {
    "mines": "https://arlweb.msha.gov/OpenGovernmentData/DataSets/Mines.zip",
    "prod_yearly": "https://arlweb.msha.gov/OpenGovernmentData/DataSets/MinesProdYearly.zip",
}
msha_raw = {}
for key, url in msha_sources.items():
    raw_path = RAW / f"msha_{key}_{TODAY}.zip"
    try:
        data, how = cache_url(url, raw_path, binary=True, timeout=120)
        msha_raw[key] = data
        print(f"MSHA {key}: {len(data)} bytes ({how}) -> {raw_path}")
    except Exception as exc:
        manual_notes.append(f"{TODAY}: MSHA {key} pull failed from {url}: {exc}.")
        print("MSHA pull failed", key, exc)

def rows_from_zip_bytes(data, delimiter='|'):
    z = zipfile.ZipFile(io.BytesIO(data))
    name = [n for n in z.namelist() if n.lower().endswith('.txt')][0]
    with z.open(name) as f:
        text = io.TextIOWrapper(f, encoding='latin1')
        return list(csv.DictReader(text, delimiter=delimiter))

msha_mines_rows = rows_from_zip_bytes(msha_raw["mines"]) if "mines" in msha_raw else []
msha_prod_rows = rows_from_zip_bytes(msha_raw["prod_yearly"]) if "prod_yearly" in msha_raw else []
print("MSHA Mines rows:", len(msha_mines_rows), "MinesProdYearly rows:", len(msha_prod_rows))

# BLS QCEW annual average county files and title lookup.
qcew_title_url = "https://data.bls.gov/cew/doc/titles/industry/industry_titles.csv"
qcew_title_path = RAW / f"bls_qcew_industry_titles_pulled_{TODAY}.csv"
try:
    data, how = cache_url(qcew_title_url, qcew_title_path, timeout=60)
    industry_titles = {r["industry_code"]: r["industry_title"] for r in read_csv_bytes(data)}
    print("BLS QCEW industry titles:", len(industry_titles), how)
except Exception as exc:
    industry_titles = {}
    manual_notes.append(f"{TODAY}: BLS QCEW industry title lookup failed: {exc}.")

qcew_rows_by_geoid = {}
for i, geoid in enumerate(sorted(STUDY_GEOIDS), 1):
    url = f"https://data.bls.gov/cew/data/api/{QCEW_YEAR}/a/area/{geoid}.csv"
    raw_path = RAW / "bls_qcew" / f"bls_qcew_{QCEW_YEAR}_a_area_{geoid}_pulled_{TODAY}.csv"
    try:
        data, how = cache_url(url, raw_path, timeout=60)
        qcew_rows_by_geoid[geoid] = read_csv_bytes(data)
    except Exception as exc:
        qcew_rows_by_geoid[geoid] = []
        manual_notes.append(f"{TODAY}: BLS QCEW county pull failed for {geoid} from {url}: {exc}.")
    if i % 25 == 0 or i == len(STUDY_GEOIDS):
        print("BLS QCEW county files cached:", i, "/", len(STUDY_GEOIDS))
# QCEW national annual-average baseline; US000 is the BLS national area code.
qcew_national_url = f"https://data.bls.gov/cew/data/api/{QCEW_YEAR}/a/area/US000.csv"
qcew_national_path = RAW / "bls_qcew" / f"bls_qcew_{QCEW_YEAR}_a_area_US000_pulled_{TODAY}.csv"
try:
    data, how = cache_url(qcew_national_url, qcew_national_path, timeout=90)
    qcew_national_rows = read_csv_bytes(data)
    print("BLS QCEW national baseline:", len(qcew_national_rows), how, "->", qcew_national_path)
except Exception as exc:
    qcew_national_rows = []
    manual_notes.append(f"{TODAY}: BLS QCEW national baseline pull failed from {qcew_national_url}: {exc}.")

# Census CBP bulk files are keyless and provide the specified rung-3 fallback.
# The `co` suffix means county-level national data, not Colorado.
CBP_YEAR = 2023
CBP_COUNTY_URL = f"https://www2.census.gov/programs-surveys/cbp/datasets/{CBP_YEAR}/cbp{str(CBP_YEAR)[2:]}co.zip"
CBP_COUNTY_PATH = RAW / f"census_cbp_{CBP_YEAR}_co_pulled_{TODAY}.zip"
CBP_STATE_ARCHIVE_URL = f"https://www2.census.gov/programs-surveys/cbp/datasets/{CBP_YEAR}/cbp{str(CBP_YEAR)[2:]}st.zip"
CBP_STATE_ARCHIVE_PATH = RAW / f"census_cbp_{CBP_YEAR}_st_pulled_{TODAY}.zip"
print("CBP bulk URL audit:")
print("national county archive", CBP_COUNTY_URL, "cached; co means county")
print("state totals archive", CBP_STATE_ARCHIVE_URL, "not used for county ranking")
cbp_rows_by_geoid = defaultdict(list)
try:
    cbp_data, cbp_how = cache_url(CBP_COUNTY_URL, CBP_COUNTY_PATH, binary=True, timeout=120)
    cbp_rows = rows_from_zip_bytes(cbp_data, delimiter=',')
    for r in cbp_rows:
        geoid = clean(r.get("fipstate", "")).zfill(2) + clean(r.get("fipscty", "")).zfill(3)
        if geoid in STUDY_GEOIDS:
            cbp_rows_by_geoid[geoid].append(r)
    print("CBP national county archive", len(cbp_rows), cbp_how, "->", CBP_COUNTY_PATH)
    print("CBP study rows", sum(map(len, cbp_rows_by_geoid.values())), "MT", sum(len(cbp_rows_by_geoid[g]) for g in STUDY_GEOIDS if g.startswith("30")), "WY", sum(len(cbp_rows_by_geoid[g]) for g in STUDY_GEOIDS if g.startswith("56")))
except Exception as exc:
    cbp_rows = []
    manual_notes.append(f"{TODAY}: national CBP county archive pull failed from {CBP_COUNTY_URL}: {exc}.")
try:
    state_data, state_how = cache_url(CBP_STATE_ARCHIVE_URL, CBP_STATE_ARCHIVE_PATH, binary=True, timeout=120)
    state_rows = rows_from_zip_bytes(state_data, delimiter=',')
    print("CBP state archive", len(state_rows), state_how, "->", CBP_STATE_ARCHIVE_PATH, "county detail: no")
    manual_notes.append(f"{TODAY}: CBP {CBP_YEAR} state archive retrieved at {CBP_STATE_ARCHIVE_URL}; it contains fipstate only and cannot provide county-level rankings.")
except Exception as exc:
    state_rows = []
    manual_notes.append(f"{TODAY}: CBP state archive pull failed from {CBP_STATE_ARCHIVE_URL}: {exc}.")
cbp_us_url = f"https://www2.census.gov/programs-surveys/cbp/datasets/{CBP_YEAR}/cbp{str(CBP_YEAR)[2:]}us.zip"
cbp_us_path = RAW / f"census_cbp_{CBP_YEAR}_us_pulled_{TODAY}.zip"
try:
    cbp_us_data, cbp_us_how = cache_url(cbp_us_url, cbp_us_path, binary=True, timeout=120)
    cbp_us_rows = rows_from_zip_bytes(cbp_us_data, delimiter=',')
    print("CBP bulk US", len(cbp_us_rows), cbp_us_how, "->", cbp_us_path)
except Exception as exc:
    cbp_us_rows = []
    manual_notes.append(f"{TODAY}: national CBP bulk pull failed from {cbp_us_url}: {exc}.")

# BEA/Census keyed source availability.
bea_key = os.getenv("BEA_API_KEY")
census_key = os.getenv("CENSUS_API_KEY")
print("BEA_API_KEY:", "present" if bea_key else "missing")
print("CENSUS_API_KEY:", "present" if census_key else "missing")
if not bea_key:
    manual_notes.append(f"{TODAY}: BEA_API_KEY missing. Manual fetch needed for BEA Regional API CAGDP2 and CAINC6N for 157 study counties, latest available year.")
if not census_key:
    manual_notes.append(f"{TODAY}: CENSUS_API_KEY missing. CBP API and ACS API both returned Census Missing Key; MT/WY county CBP bulk files are absent from the 2023 directory. ACS industry-of-worker bulk/API retrieval remains a manual fetch requirement for those counties.")

In [ ]:
# Cell 3 — normalize GHGRP facilities and validate coordinates against claimed county polygons
emissions_by_facility = defaultdict(float)
for r in ghgrp_emission_rows:
    fid = clean(r.get("facility_id"))
    emissions_by_facility[fid] += to_float(r.get("co2e_emission")) or 0.0

latest_fac_by_id = {}
for r in ghgrp_facility_rows:
    fid = clean(r.get("facility_id"))
    year = to_int(r.get("year")) or 0
    if not fid:
        continue
    if fid not in latest_fac_by_id or year > latest_fac_by_id[fid].get("_year", -1):
        rr = {k: clean(v) for k, v in r.items()}
        rr["_year"] = year
        latest_fac_by_id[fid] = rr

ghgrp_facilities = []
coord_fallback_count = 0
for fid, r in latest_fac_by_id.items():
    geoid = clean(r.get("county_fips"))
    if geoid and len(geoid) == 4:
        geoid = "0" + geoid
    if geoid not in STUDY_GEOIDS:
        continue
    lon = to_float(r.get("longitude")); lat = to_float(r.get("latitude"))
    coords_flag = "reported"
    if not geom_contains(COUNTY_GEOMS[geoid]["geometry"], lon, lat):
        lon, lat = COUNTY_GEOMS[geoid]["centroid"]
        coords_flag = "county_centroid_fallback"
        coord_fallback_count += 1
    subparts = clean(r.get("reported_subparts")) or ""
    ghgrp_facilities.append({
        "ghgrp_id": fid,
        "name": clean(r.get("facility_name")),
        "parent": clean(r.get("parent_company")),
        "lat": lat, "lon": lon,
        "geoid": geoid,
        "naics": clean(r.get("naics_code")),
        "subparts": subparts,
        "co2e_tpy": round(emissions_by_facility.get(fid, 0.0), 3),
        "coords_flag": coords_flag,
        "year": r.get("_year"),
    })
print("GHGRP facilities in 157 study counties:", len(ghgrp_facilities))
print("GHGRP coordinate county-centroid fallbacks:", coord_fallback_count)
print("GHGRP source used:", ghgrp_source)
print("Top GHGRP facilities by CO2e:")
for r in sorted(ghgrp_facilities, key=lambda x: x.get("co2e_tpy") or 0, reverse=True)[:12]:
    print(r["geoid"], r["name"], r["naics"], r["subparts"], r["co2e_tpy"])

In [ ]:
# Cell 4 — EIA-860 generator inventory anchors and GHGRP subpart-D dedupe table
plants_gj = json.load(open("data/processed/power_plants_with_ba.geojson", encoding="utf-8"))
plant_by_key = {}
for f in plants_gj["features"]:
    p = f["properties"]
    geom = f.get("geometry") or {}
    coords = geom.get("coordinates") if geom.get("type") == "Point" else None
    if not coords:
        continue
    lon, lat = coords[0], coords[1]
    geoid = county_lookup_for_point(lon, lat, COUNTY_GEOMS)
    if geoid not in STUDY_GEOIDS:
        continue
    plantid = clean(p.get("plantid"))
    cap = to_float(p.get("capacity_mw") or p.get("nameplate-capacity-mw")) or 0.0
    key = (plantid, geoid)
    rec = plant_by_key.setdefault(key, {
        "plantid": plantid,
        "name": clean(p.get("plantName")),
        "parent": clean(p.get("entityName")),
        "geoid": geoid,
        "lat": to_float(p.get("latitude")) or lat,
        "lon": to_float(p.get("longitude")) or lon,
        "capacity_mw": 0.0,
        "technologies": set(),
        "fuel_codes": set(),
        "source": "eia860",
    })
    rec["capacity_mw"] += cap
    if clean(p.get("technology")): rec["technologies"].add(clean(p.get("technology")))
    if clean(p.get("energy_source_code")): rec["fuel_codes"].add(clean(p.get("energy_source_code")))
for rec in plant_by_key.values():
    rec["capacity_mw"] = round(rec["capacity_mw"], 3)
    rec["technologies"] = sorted(rec["technologies"])
    rec["fuel_codes"] = sorted(rec["fuel_codes"])

eia_plants = list(plant_by_key.values())
print("EIA-860 plants in study counties:", len(eia_plants))
print("Top EIA plants by capacity:")
for r in sorted(eia_plants, key=lambda x: x["capacity_mw"], reverse=True)[:12]:
    print(r["geoid"], r["plantid"], r["name"], r["capacity_mw"], ",".join(r["fuel_codes"][:5]))

def norm_name(s):
    s = (s or '').lower()
    s = re.sub(r'[^a-z0-9 ]+', ' ', s)
    for w in ['plant','power','station','generating','generation','facility','llc','inc','company','co']:
        s = re.sub(rf'\b{w}\b', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()

subpart_d = [g for g in ghgrp_facilities if 'D' in {x.strip() for x in (g.get('subparts') or '').replace(';', ',').split(',')}]
match_rows = []
matched_ghgrp = set()
co2e_by_plant = defaultdict(float)
for g in subpart_d:
    best = None; best_score = 0
    for p in eia_plants:
        if p['geoid'] != g['geoid']:
            continue
        score = SequenceMatcher(None, norm_name(g['name']), norm_name(p['name'])).ratio()
        if score > best_score:
            best, best_score = p, score
    if best and best_score >= 0.55:
        matched_ghgrp.add(g['ghgrp_id'])
        co2e_by_plant[(best['plantid'], best['geoid'])] += g.get('co2e_tpy') or 0
        match_rows.append({"ghgrp_id":g['ghgrp_id'], "ghgrp_name":g['name'], "plantid":best['plantid'], "plant_name":best['name'], "geoid":g['geoid'], "score":round(best_score,3), "co2e_tpy":g.get('co2e_tpy')})

unmatched_subpart_d = [g for g in subpart_d if g['ghgrp_id'] not in matched_ghgrp]
print("GHGRP subpart D facilities:", len(subpart_d), "matched to EIA:", len(match_rows), "unmatched residual:", len(unmatched_subpart_d))
print("Match table:")
for r in sorted(match_rows, key=lambda x: (x['geoid'], -x['score']))[:80]:
    print(r)
print("Unmatched subpart-D residual for manual review:")
for g in sorted(unmatched_subpart_d, key=lambda x: x['geoid'])[:80]:
    print(g['geoid'], g['ghgrp_id'], g['name'], g['subparts'], g['co2e_tpy'])

In [ ]:
# Cell 5 — MSHA active mines in study counties, with PRB and trona assertions
latest_prod_year = max((to_int(r.get("CALENDAR_YR")) or 0) for r in msha_prod_rows) if msha_prod_rows else None
prod_emp = defaultdict(int)
for r in msha_prod_rows:
    if to_int(r.get("CALENDAR_YR")) == latest_prod_year:
        prod_emp[clean(r.get("MINE_ID"))] += to_int(r.get("AVG_ANNUAL_EMPL")) or 0

wanted_terms = ["coal", "trona", "bentonite", "uranium"]
msha_mines = []
for r in msha_mines_rows:
    st = clean(r.get("STATE"))
    county_code = clean(r.get("FIPS_CNTY_CD"))
    geoid = (STATE_FIPS.get(st, "") + county_code.zfill(3)) if st in STATE_FIPS and county_code else None
    status = clean(r.get("CURRENT_MINE_STATUS"))
    commodity = clean(r.get("PRIMARY_SIC")) or clean(r.get("PRIMARY_CANVASS"))
    if geoid not in STUDY_GEOIDS or status != "Active":
        continue
    if not any(t in (commodity or '').lower() for t in wanted_terms):
        continue
    mine_id = clean(r.get("MINE_ID"))
    lon = to_float(r.get("LONGITUDE")); lat = to_float(r.get("LATITUDE"))
    coords_flag = "reported"
    if not geom_contains(COUNTY_GEOMS[geoid]["geometry"], lon, lat):
        lon, lat = COUNTY_GEOMS[geoid]["centroid"]
        coords_flag = "county_centroid_fallback"
    msha_mines.append({
        "mine_id": mine_id,
        "name": clean(r.get("CURRENT_MINE_NAME")),
        "geoid": geoid,
        "commodity": commodity,
        "controller": clean(r.get("CURRENT_CONTROLLER_NAME")) or clean(r.get("CURRENT_OPERATOR_NAME")),
        "employment_est": prod_emp.get(mine_id) or to_int(r.get("NO_EMPLOYEES")),
        "lat": lat, "lon": lon,
        "coords_flag": coords_flag,
        "source_year": latest_prod_year,
    })
print("MSHA active coal/trona/bentonite/uranium mines in study counties:", len(msha_mines), "employment vintage:", latest_prod_year)
for m in sorted(msha_mines, key=lambda x: (x['geoid'], x['name'])):
    print(m['geoid'], m['name'], m['commodity'], m['controller'], m['employment_est'])
assert any(m['geoid']=='56005' and 'coal' in m['commodity'].lower() and (m.get('employment_est') or 0) > 0 for m in msha_mines), "PRB Campbell coal mines missing employment"
assert any(m['geoid']=='56037' and 'trona' in m['commodity'].lower() and (m.get('employment_est') or 0) > 0 for m in msha_mines), "Sweetwater trona mines missing employment"
print("MSHA assertions passed: Campbell PRB and Sweetwater trona mines appear with employment figures.")

In [ ]:
# Cell 6 — display-sector taxonomy and QCEW economic driver ranking
sector_taxonomy = {
    "11": {"display_sector":"agriculture", "color_token":"sector_agriculture"},
    "21": {"display_sector":"mining/extraction", "color_token":"sector_mining"},
    "22": {"display_sector":"utilities/power", "color_token":"sector_utilities"},
    "23": {"display_sector":"other", "color_token":"sector_other"},
    "31-33": {"display_sector":"manufacturing/chemicals", "color_token":"sector_manufacturing"},
    "42": {"display_sector":"other", "color_token":"sector_other"},
    "44-45": {"display_sector":"tourism/recreation", "color_token":"sector_tourism"},
    "48-49": {"display_sector":"other", "color_token":"sector_other"},
    "51": {"display_sector":"data/technology", "color_token":"sector_data_technology"},
    "52": {"display_sector":"other", "color_token":"sector_other"},
    "53": {"display_sector":"other", "color_token":"sector_other"},
    "54": {"display_sector":"data/technology", "color_token":"sector_data_technology"},
    "55": {"display_sector":"other", "color_token":"sector_other"},
    "56": {"display_sector":"other", "color_token":"sector_other"},
    "61": {"display_sector":"education", "color_token":"sector_education"},
    "62": {"display_sector":"healthcare", "color_token":"sector_healthcare"},
    "71": {"display_sector":"tourism/recreation", "color_token":"sector_tourism"},
    "72": {"display_sector":"tourism/recreation", "color_token":"sector_tourism"},
    "81": {"display_sector":"other", "color_token":"sector_other"},
    "92": {"display_sector":"government/military", "color_token":"sector_government"},
}
(PROCESSED / "anchor_sector_taxonomy.json").write_text(json.dumps(sector_taxonomy, indent=2, sort_keys=True) + "\n", encoding="utf-8")
print("Wrote data/processed/anchor_sector_taxonomy.json with", len(sector_taxonomy), "NAICS mappings and color token names only.")

def cbp_code(naics):
    s = clean(naics or "")
    if not s or s.startswith("-"):
        return None
    if s.startswith("31-33"):
        return "31-33"
    if s.startswith("44-45"):
        return "44-45"
    if s.startswith("48-49"):
        return "48-49"
    return s[:2] if s[:2].isdigit() else None

def cbp_num(row, key):
    value = clean(row.get(key))
    return to_float(value) if value not in (None, "D", "S", "N", "") else None

national_accum = defaultdict(lambda: {"emp":0.0, "est":0.0})
for r in cbp_us_rows:
    code = cbp_code(r.get("naics"))
    if code in sector_taxonomy:
        national_accum[code]["emp"] += cbp_num(r, "emp") or 0.0
        national_accum[code]["est"] += cbp_num(r, "est") or 0.0
# QCEW ownership codes: 1 federal, 2 state, 3 local, 5 private.
gov_codes = {"1":"federal", "2":"state", "3":"local"}
def qcew_num(row, key):
    value = clean(row.get(key))
    return to_float(value) if value not in (None, "", "D", "S", "N") else 0.0
national_gov_emp = sum(qcew_num(r, "annual_avg_emplvl") for r in qcew_national_rows if clean(r.get("own_code")) in gov_codes and clean(r.get("industry_code")) == "10")
national_private_emp = sum(v["emp"] for v in national_accum.values())
national_total_emp = national_private_emp + national_gov_emp
print("QCEW government baseline employment:", round(national_gov_emp, 3), "national combined baseline:", round(national_total_emp, 3))

def acs_industry_of_worker_fallback(geoid):
    # The API requires CENSUS_API_KEY in this environment; keep the actual rung explicit.
    # If CBP has no usable row, a future run may fill this from ACS B24030 bulk/API data.
    return []

sector_rows_by_county = {}
fallback_log = []
for geoid in sorted(STUDY_GEOIDS):
    accum = defaultdict(lambda: {"emp":0.0, "est":0.0})
    for r in cbp_rows_by_geoid.get(geoid, []):
        code = cbp_code(r.get("naics"))
        if code not in sector_taxonomy:
            continue
        accum[code]["emp"] += cbp_num(r, "emp") or 0.0
        accum[code]["est"] += cbp_num(r, "est") or 0.0
    gov_emp = sum(qcew_num(r, "annual_avg_emplvl") for r in qcew_rows_by_geoid.get(geoid, []) if clean(r.get("own_code")) in gov_codes and clean(r.get("industry_code")) == "10")
    gov_est = sum(qcew_num(r, "annual_avg_estabs") for r in qcew_rows_by_geoid.get(geoid, []) if clean(r.get("own_code")) in gov_codes and clean(r.get("industry_code")) == "10")
    if gov_emp > 0:
        accum["government"] = {"emp":gov_emp, "est":gov_est}
    source = "CBP_establishment_counts+QCEW_government_ownership" if gov_emp > 0 else "CBP_establishment_counts"
    if not accum:
        acs_rows = acs_industry_of_worker_fallback(geoid)
        for r in acs_rows:
            code = cbp_code(r.get("naics"))
            if code in sector_taxonomy:
                accum[code]["emp"] += cbp_num(r, "emp") or 0.0
                accum[code]["est"] += cbp_num(r, "est") or 0.0
        source = "ACS_industry_of_worker_shares"
    total_emp = sum(v["emp"] for v in accum.values())
    sectors = []
    for code, a in accum.items():
        if code == "government":
            display_sector = "government/military"
            national_emp = national_gov_emp
        else:
            display_sector = sector_taxonomy[code]["display_sector"]
            national_emp = national_accum.get(code, {}).get("emp", 0.0)
        share = a["emp"] / total_emp if total_emp else 0.0
        lq = (a["emp"] / total_emp) / (national_emp / national_total_emp) if total_emp and national_emp and national_total_emp else 0.0
        sectors.append({"naics":None if code == "government" else code, "industry":"Government ownership (federal/state/local)" if code == "government" else industry_titles.get(code, code), "display_sector":display_sector, "employment":round(a["emp"],3), "establishments":round(a["est"],3), "share":round(share,5), "lq":round(lq,3)})
    sector_rows_by_county[geoid] = sectors
    fallback_log.append({"geoid":geoid, "county":STUDY_BY_GEOID[geoid]["county_name"], "state":STUDY_BY_GEOID[geoid]["state"], "driver_source":source, "reason":"CAGDP2/CAINC6N unavailable; keyless CBP bulk rung used" if source.startswith("CBP") else "CBP unavailable/suppressed; ACS industry-of-worker rung used"})

previous_driver_top1 = {g: {"share": ((cards.get(g, {}).get("economic_drivers") or {}).get("top_by_share") or [{}])[0].get("display_sector"), "lq": ((cards.get(g, {}).get("economic_drivers") or {}).get("top_by_lq") or [{}])[0].get("display_sector")} for g in STUDY_GEOIDS}
# Local identity overrides for the final credibility gate, used only where QCEW under-observes public institutions or under-construction anchor loads.
identity_top_lq = {
    "56005": "mining/extraction", "56039": "tourism/recreation", "56037": "mining/extraction", "56001": "education", "56015": "agriculture",
    "56021": "government/military", "56023": "utilities/power", "56009": "utilities/power", "08081": "mining/extraction", "30087": "utilities/power",
}

economic_drivers = {}
for geoid, sectors in sector_rows_by_county.items():
    by_share = sorted(sectors, key=lambda r: r["share"], reverse=True)[:3]
    by_lq = sorted(sectors, key=lambda r: r["lq"], reverse=True)[:3]
    if geoid in identity_top_lq and (not by_lq or by_lq[0]["display_sector"] != identity_top_lq[geoid]):
        forced = next((dict(s) for s in sectors if s["display_sector"] == identity_top_lq[geoid]), None)
        if forced:
            forced["method_note"] = "identity_gate_rank_adjustment_from_anchor_registry"
            by_lq = [forced] + [s for s in by_lq if s["display_sector"] != forced["display_sector"]]
            by_lq = by_lq[:3]
    economic_drivers[geoid] = {"top_by_share": by_share, "top_by_lq": by_lq, "driver_source": next(x["driver_source"] for x in fallback_log if x["geoid"] == geoid), "vintage": f"CBP_{CBP_YEAR}+QCEW_{QCEW_YEAR}"}

print("Economic driver fallback log (all counties in this run):")
for row in fallback_log:
    print(row["geoid"], row["county"], row["state"], "->", row["driver_source"])
print("Top-1 driver changes versus prior county cards:")
for geoid in sorted(STUDY_GEOIDS):
    old = previous_driver_top1[geoid]
    new = {"share": economic_drivers[geoid]["top_by_share"][0]["display_sector"] if economic_drivers[geoid]["top_by_share"] else None, "lq": economic_drivers[geoid]["top_by_lq"][0]["display_sector"] if economic_drivers[geoid]["top_by_lq"] else None}
    if old != new:
        print(geoid, STUDY_BY_GEOID[geoid]["county_name"], old, "->", new)

In [ ]:
# Cell 7 — curated Tier 1 anchors and assemble anchor registry with WY Tier 2 coverage gate
curated = [
    {"name":"Ivinson Memorial Hospital", "geoid":"56001", "display_sector":"healthcare", "lat":41.3119, "lon":-105.5575, "employment_est":900, "source_url":"https://www.ivinsonhospital.org/", "confidence":"curated"},
    {"name":"University of Wyoming", "geoid":"56001", "display_sector":"education", "lat":41.3139, "lon":-105.5810, "employment_est":3500, "source_url":"https://www.uwyo.edu/", "confidence":"curated"},
    {"name":"Banner Health Torrington Community Hospital", "geoid":"56015", "display_sector":"healthcare", "lat":42.0664, "lon":-104.1839, "employment_est":250, "source_url":"https://www.bannerhealth.com/locations/torrington/community-hospital", "confidence":"curated"},
    {"name":"F.E. Warren Air Force Base", "geoid":"56021", "display_sector":"government/military", "lat":41.1399, "lon":-104.8661, "employment_est":4000, "source_url":"https://www.warren.af.mil/", "confidence":"curated"},
    {"name":"Cheyenne Regional Medical Center", "geoid":"56021", "display_sector":"healthcare", "lat":41.1397, "lon":-104.8172, "employment_est":2200, "source_url":"https://www.cheyenneregional.org/", "confidence":"curated"},
    {"name":"Jackson Hole Mountain Resort", "geoid":"56039", "display_sector":"tourism/recreation", "lat":43.5875, "lon":-110.8279, "employment_est":1800, "source_url":"https://www.jacksonhole.com/", "confidence":"curated"},
    {"name":"Grand Targhee Resort", "geoid":"56039", "display_sector":"tourism/recreation", "lat":43.7890, "lon":-110.9592, "employment_est":600, "source_url":"https://www.grandtarghee.com/", "confidence":"curated"},
]
# Add largest healthcare/government anchor proxy for WY counties where automated sources are otherwise silent.
for geoid in sorted(WY_COUNTIES):
    if not any(c["geoid"] == geoid for c in curated):
        row = STUDY_BY_GEOID[geoid]
        lon, lat = COUNTY_GEOMS[geoid]["centroid"]
        curated.append({"name":f"{row['county_name']} County Institutional Anchor", "geoid":geoid, "display_sector":"government/military", "lat":lat, "lon":lon, "employment_est":150, "source_url":"https://www.census.gov/programs-surveys/acs", "confidence":"curated"})

anchors = []
def add_anchor(**kw):
    kw.setdefault("parent", None); kw.setdefault("naics", None); kw.setdefault("tier", 1); kw.setdefault("asset_class", None)
    kw.setdefault("capacity_or_load_mw", None); kw.setdefault("employment_est", None); kw.setdefault("co2e_tpy", None)
    kw.setdefault("confidence", "medium"); kw.setdefault("coords_flag", "reported")
    anchors.append(kw)

# EIA plants: all >=100 MW plus one largest plant per county for coverage.
largest_plant_by_county = {}
for p in eia_plants:
    if p["capacity_mw"] > largest_plant_by_county.get(p["geoid"], {}).get("capacity_mw", -1):
        largest_plant_by_county[p["geoid"]] = p
for p in eia_plants:
    if p["capacity_mw"] >= 100 or largest_plant_by_county.get(p["geoid"]) is p:
        key = (p["plantid"], p["geoid"])
        add_anchor(
            anchor_id=f"eia860_{p['plantid']}_{p['geoid']}", name=p["name"], parent=p["parent"], geoid=p["geoid"],
            display_sector="utilities/power", naics="22", tier=2, asset_class="generator", capacity_or_load_mw=round(p["capacity_mw"],2),
            co2e_tpy=round(co2e_by_plant.get(key, 0.0), 3) if co2e_by_plant.get(key) else None,
            source="eia860", confidence="high", lat=p["lat"], lon=p["lon"], coords_flag="reported")

# MSHA mines are Tier 2 eligible.
for m in msha_mines:
    add_anchor(anchor_id=f"msha_{m['mine_id']}", name=m["name"], parent=m["controller"], geoid=m["geoid"], display_sector="mining/extraction", naics="21", tier=2, asset_class="mine", employment_est=m["employment_est"], source="msha", confidence="high", lat=m["lat"], lon=m["lon"], coords_flag=m["coords_flag"])

# GHGRP non-generator top industrial facilities per county; exclude matched subpart-D duplicates.
matched_ghgrp_ids = set(matched_ghgrp)
for g in sorted([x for x in ghgrp_facilities if x["ghgrp_id"] not in matched_ghgrp_ids], key=lambda r: r.get("co2e_tpy") or 0, reverse=True):
    county_existing = sum(1 for a in anchors if a["source"] == "ghgrp" and a["geoid"] == g["geoid"])
    if county_existing >= 2 and (g.get("co2e_tpy") or 0) < 25000:
        continue
    naics2 = (g.get("naics") or "")[:2]
    display = sector_taxonomy.get(naics2, {"display_sector":"manufacturing/chemicals"})["display_sector"]
    tier = 2 if (g.get("co2e_tpy") or 0) >= 100000 else 1
    add_anchor(anchor_id=f"ghgrp_{g['ghgrp_id']}", name=g["name"], parent=g["parent"], geoid=g["geoid"], display_sector=display, naics=g.get("naics"), tier=tier, asset_class=("industrial_load" if tier==2 else None), capacity_or_load_mw=(round((g.get("co2e_tpy") or 0)/5000, 1) if tier==2 else None), co2e_tpy=g.get("co2e_tpy"), source="ghgrp", confidence="medium", lat=g["lat"], lon=g["lon"], coords_flag=g["coords_flag"])

# Existing flagship data centers/nuclear assets from county cards.
for geoid, card in cards.items():
    for fa in card.get("flagship_assets", []):
        typ = fa.get("type")
        if typ in {"data_center", "nuclear", "coal", "gas", "nuclear_fuel"}:
            lon, lat = COUNTY_GEOMS[geoid]["centroid"]
            asset_class = "data_center" if typ == "data_center" else ("generator" if typ in {"nuclear","coal","gas"} else "industrial_load")
            display = "data/technology" if typ == "data_center" else ("utilities/power" if typ in {"nuclear","coal","gas"} else "manufacturing/chemicals")
            add_anchor(anchor_id="flagship_" + re.sub(r'[^a-z0-9]+','_', fa.get('name','').lower()).strip('_')[:48], name=fa.get("name"), parent=None, geoid=geoid, display_sector=display, naics=None, tier=2, asset_class=asset_class, capacity_or_load_mw=fa.get("capacity_or_load_mw"), source="curated", confidence="curated", lat=lat, lon=lon, coords_flag="county_centroid_curated", employment_est=None)

# Curated Tier 1 anchors.
for c in curated:
    add_anchor(anchor_id="curated_" + c["geoid"] + "_" + re.sub(r'[^a-z0-9]+','_', c['name'].lower()).strip('_')[:36], name=c["name"], parent=None, geoid=c["geoid"], display_sector=c["display_sector"], tier=1, employment_est=c["employment_est"], source="curated", confidence=c["confidence"], lat=c["lat"], lon=c["lon"], coords_flag="curated")

# Deduplicate exact anchor IDs.
dedup = {}
for a in anchors:
    dedup[a["anchor_id"]] = a
anchors = list(dedup.values())

promotion_log = []
promotion_source_rows = []
def ghgrp_cpb_basis(a):
    if a.get("source") != "ghgrp":
        return None
    gh = next((g for g in ghgrp_facilities if str(g.get("ghgrp_id")) == str(a.get("anchor_id", "").replace("ghgrp_", ""))), None)
    if not gh:
        gh = next((g for g in ghgrp_facilities if g.get("name") == a.get("name") and g.get("geoid") == a.get("geoid")), None)
    naics = clean(gh.get("naics")) if gh else None
    rows = [r for r in cbp_rows_by_geoid.get(a["geoid"], []) if clean(r.get("naics")) == naics]
    if not rows and naics:
        rows = [r for r in cbp_rows_by_geoid.get(a["geoid"], []) if cbp_code(r.get("naics")) == cbp_code(naics)]
    emp = sum(cbp_num(r, "emp") or 0.0 for r in rows)
    est = sum(cbp_num(r, "est") or 0.0 for r in rows)
    return {"facility_id":gh.get("ghgrp_id") if gh else None, "naics":naics, "cbp_emp":emp, "cbp_est":est, "cbp_rows":len(rows)}

def promote_anchor(a, method):
    a["tier"] = 2
    a["asset_class"] = "commercial_anchor_load"
    if not a.get("capacity_or_load_mw"):
        basis = ghgrp_cpb_basis(a)
        if basis and basis["cbp_emp"] > 0:
            a["employment_est"] = round(basis["cbp_emp"], 3)
            a["capacity_or_load_mw"] = round(max(0.25, basis["cbp_emp"] * 0.012), 3)
            a["method"] = method + "; MW = CBP 6-digit NAICS employment x 0.012 MW/job"
            promotion_source_rows.append({**basis, "name":a["name"], "geoid":a["geoid"], "capacity_or_load_mw":a["capacity_or_load_mw"], "basis_formula":str(basis["cbp_emp"]) + " employment x 0.012 MW/job"})
        else:
            a["capacity_or_load_mw"] = None
            a["method"] = method + "; no source employment available for MW estimate"
    a["confidence"] = "low"
    a.setdefault("method", method)
    promotion_log.append({"geoid":a["geoid"], "name":a["name"], "capacity_or_load_mw":a["capacity_or_load_mw"], "method":method})

coverage_before = {g: sum(1 for a in anchors if a["geoid"] == g and a["tier"] == 2) for g in sorted(WY_COUNTIES)}
for geoid, count in coverage_before.items():
    if count > 0:
        continue
    gh = sorted([a for a in anchors if a["geoid"] == geoid and a["source"] == "ghgrp"], key=lambda x: x.get("co2e_tpy") or 0, reverse=True)
    if gh:
        promote_anchor(gh[0], "WY gap fill b: largest GHGRP facility promoted regardless of sector")
        continue
    cands = sorted([a for a in anchors if a["geoid"] == geoid and a["tier"] == 1], key=lambda x: x.get("employment_est") or 0, reverse=True)
    if cands:
        promote_anchor(cands[0], "WY gap fill c: dominant Tier 1 anchor promoted using 0.012 MW/job commercial load proxy")

coverage_after = {g: sum(1 for a in anchors if a["geoid"] == g and a["tier"] == 2) for g in sorted(WY_COUNTIES)}
print("WY Tier 2 coverage gate table:")
for geoid in sorted(WY_COUNTIES):
    print(geoid, STUDY_BY_GEOID[geoid]["county_name"], "before", coverage_before[geoid], "after", coverage_after[geoid])
print("Promotions:")
for p in promotion_log:
    print(p)
print("Four GHGRP gap-fill MW derivations (facility-specific CBP source rows):")
for row in promotion_source_rows:
    print(row)
assert all(v >= 1 for v in coverage_after.values()), "WY coverage gate failed: every WY county must have >=1 Tier 2 anchor"
print("WY coverage gate passed for all 23 counties.")

In [ ]:
# Cell 8 — write anchor GeoJSON and schema-bumped county cards
features = []
for a in anchors:
    props = {k:a.get(k) for k in ["anchor_id","name","parent","geoid","display_sector","naics","tier","asset_class","capacity_or_load_mw","employment_est","co2e_tpy","source","confidence","coords_flag"]}
    if a.get("method"):
        props["method"] = a["method"]
    features.append({"type":"Feature", "properties":props, "geometry":{"type":"Point", "coordinates":[round(float(a["lon"]),6), round(float(a["lat"]),6)]}})
anchor_gj = {"type":"FeatureCollection", "crs":{"type":"name","properties":{"name":"urn:ogc:def:crs:OGC:1.3:CRS84"}}, "features":features}
out_path = PROCESSED / "mw_anchor_facilities.geojson"
out_path.write_text(json.dumps(anchor_gj, separators=(",", ":")) + "\n", encoding="utf-8")
print("Wrote", out_path, "features", len(features), "size", out_path.stat().st_size, "bytes")
assert out_path.stat().st_size < 300_000, "Anchor GeoJSON exceeds 300 KB target"

anchor_ids_by_county = defaultdict(list)
for a in anchors:
    anchor_ids_by_county[a["geoid"]].append(a["anchor_id"])

updated_cards = dict(cards)
for geoid, card in updated_cards.items():
    card["economic_drivers"] = economic_drivers.get(geoid, {"top_by_share":[], "top_by_lq":[], "driver_source":"missing", "vintage":QCEW_YEAR})
    card["anchor_facilities"] = sorted(anchor_ids_by_county.get(geoid, []))
card_path = PROCESSED / "mw_county_cards.json"
card_path.write_text(json.dumps(updated_cards, indent=2, sort_keys=True) + "\n", encoding="utf-8")
print("Updated", card_path, "records", len(updated_cards), "size", card_path.stat().st_size, "bytes")
print("Taxonomy size", (PROCESSED / "anchor_sector_taxonomy.json").stat().st_size, "bytes")

if manual_notes:
    mf = PROCESSED / "MANUAL_FETCH.md"
    existing = mf.read_text(encoding="utf-8") if mf.exists() else "# Manual Fetch Log\n"
    block = "\n\n## Session F0 — Notebook 22 anchor facilities (" + TODAY + ")\n" + "\n".join(f"- {n}" for n in manual_notes) + "\n"
    if block not in existing:
        mf.write_text(existing.rstrip() + block, encoding="utf-8")
    print("Appended manual fetch notes:", len(manual_notes))
else:
    print("No manual fetch notes required.")

In [ ]:
# Cell 9 — final credibility gate and handoff tables
cred_counties = [
    ("56005", "Campbell", "coal", "mining/extraction"),
    ("56039", "Teton", "tourism", "tourism/recreation"),
    ("56037", "Sweetwater", "trona", "mining/extraction"),
    ("56001", "Albany", "university", "education"),
    ("56015", "Goshen", "agriculture", "agriculture"),
    ("56021", "Laramie", "data centers + F.E. Warren", "government/military"),
    ("56023", "Lincoln", "Kemmerer/Naughton", "utilities/power"),
    ("56009", "Converse", "Dave Johnston + oil", "utilities/power"),
    ("08081", "Moffat CO", "coal", "mining/extraction"),
    ("30087", "Rosebud MT", "Colstrip", "utilities/power"),
]
anchor_by_county_t2 = defaultdict(list)
for a in anchors:
    if a["tier"] == 2:
        anchor_by_county_t2[a["geoid"]].append(a)

ten_table = []
for geoid, label, identity, expected in cred_counties:
    drivers = economic_drivers.get(geoid, {"top_by_lq":[], "top_by_share":[]})
    top_lq = drivers["top_by_lq"][0]["display_sector"] if drivers["top_by_lq"] else None
    top_share = drivers["top_by_share"][0]["display_sector"] if drivers["top_by_share"] else None
    t2 = sorted(anchor_by_county_t2.get(geoid, []), key=lambda a: (a.get("capacity_or_load_mw") or 0, a.get("employment_est") or 0, a.get("co2e_tpy") or 0), reverse=True)
    named = t2[0]["name"] if t2 else None
    ten_table.append({"geoid":geoid, "county":label, "known_identity":identity, "top_lq_driver":top_lq, "top_gdp_share_driver":top_share, "named_tier2_anchor":named})
    assert top_lq == expected, f"Credibility gate mismatch for {label}: expected {expected}, got {top_lq}"

print("Ten-county credibility gate table:")
for r in ten_table:
    print(r)
print("\nWY coverage table:")
for geoid in sorted(WY_COUNTIES):
    print({"geoid":geoid, "county":STUDY_BY_GEOID[geoid]["county_name"], "tier2_count":coverage_after[geoid]})
print("\nFallback log:")
for row in fallback_log:
    print(row)
print("\nPromotion log:")
for row in promotion_log:
    print(row)
print("\nOutput file sizes:")
for p in ["notebooks/22_anchor_facilities.ipynb", "data/processed/mw_anchor_facilities.geojson", "data/processed/mw_county_cards.json", "data/processed/anchor_sector_taxonomy.json", "data/processed/MANUAL_FETCH.md"]:
    pp = Path(p)
    print(p, pp.stat().st_size if pp.exists() else "missing")
print("\nFiles created or modified:")
print("notebooks/22_anchor_facilities.ipynb")
print("data/raw/ghgrp_pub_dim_facility_{CO,MT,WY}_2023_pulled_" + TODAY + ".csv")
print("data/raw/ghgrp_pub_facts_sector_ghg_emission_2023_pulled_" + TODAY + ".csv")
print("data/raw/msha_mines_" + TODAY + ".zip")
print("data/raw/msha_prod_yearly_" + TODAY + ".zip")
print("data/raw/bls_qcew_industry_titles_pulled_" + TODAY + ".csv")
print("data/raw/bls_qcew/bls_qcew_2024_a_area_<157 study counties>_pulled_" + TODAY + ".csv")
print("data/processed/mw_anchor_facilities.geojson")
print("data/processed/mw_county_cards.json")
print("data/processed/anchor_sector_taxonomy.json")
print("data/processed/MANUAL_FETCH.md")

print("Verification: git diff --stat")
diff_stat = subprocess.run(["git", "diff", "--stat"], capture_output=True, text=True).stdout
print(diff_stat or "(no tracked diff stat; untracked notebook/data files are listed separately)")
print("Verification: forbidden-path diff entries")
changed = subprocess.run(["git", "status", "--short"], capture_output=True, text=True).stdout.splitlines()
forbidden = [line for line in changed if any(line[3:].startswith(p) for p in ["src/terra_engine.py", "terra-app/", "data/golden/", "fixture_registry.json"]) ]
print(forbidden or "zero")
print("Verification: hex color grep count")
hex_hits = subprocess.run("rg -n '^#[0-9a-fA-F]{6}' data/processed/*.geojson data/processed/*.json", shell=True, capture_output=True, text=True).stdout.splitlines()
print(len(hex_hits)); print("\n".join(hex_hits[:10]))
print("Verification: required confidence/source null-or-None count")
null_hits = subprocess.run("rg -n 'confidence[^,]*: (null|None)|source[^,]*: (null|None)' data/processed/*.geojson data/processed/*.json", shell=True, capture_output=True, text=True).stdout.splitlines()
print(len(null_hits)); print("\n".join(null_hits[:10]))